## Part 1 - Cleaning

In [ ]:
import numpy as np
import pandas as pd

orderdf = pd.read_csv("D:/Downloads/Team Project/0_Data/olist_orders_dataset.csv")

# Check the null values
orderdf.isnull().sum()

# Maybe we have null dates because the orders were are still in process in the time we extracted the data 

# Either ways we are going to remove the rows that contain null value in this case
orderdf.dropna(inplace=True)

In [ ]:
# Convert Date Columns to Datetime
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orderdf[col] = pd.to_datetime(orderdf[col], errors='coerce')

# Check again
orderdf.info()

In [ ]:
# Load product table
productdf = pd.read_csv("D:/Downloads/Team Project/0_Data/olist_products_dataset.csv")

# Check table
productdf.info()

# Check null values
productdf.isnull().sum()

# Since we only care about category name we are going to remove any row that have no category name
productdf.dropna(subset=["product_category_name"],inplace=True)

In [ ]:
# Load the translation table
translatondf = pd.read_csv("D:/Downloads/Team Project/0_Data/product_category_name_translation.csv")

# Merge Both Tables
productdf = pd.merge(productdf, translatondf, on='product_category_name', how='left')

# Replace columns (we add fillna in case if the cell ran again so it wont destruct the data)
productdf['product_category_name'] = productdf['product_category_name_english'].fillna(productdf['product_category_name'])

# Drop product_category_name_english column
productdf.drop(columns=["product_category_name_english"],inplace=True)

# Handle missing translations
productdf['product_category_name'] = productdf['product_category_name'].fillna('other')

# Alter the column
productdf["product_category_name"] = productdf["product_category_name"].str.replace("_"," ").str.title()

# Check the unique categorys
productdf["product_category_name"].unique()

In [ ]:
# Load order items table
itemsdf = pd.read_csv("D:/Downloads/Team Project/0_Data/olist_order_items_dataset.csv")

# Check table
itemsdf.info()

# Convert shipping_limit_date column to datetime data type
itemsdf["shipping_limit_date"] = pd.to_datetime(itemsdf["shipping_limit_date"])

# Check null values
itemsdf.isnull().sum()

In [ ]:
# Load customers table
customersdf = pd.read_csv("D:/Downloads/Team Project/0_Data/olist_customers_dataset.csv")

# Check table
customersdf.info()

# Check null values
customersdf.isnull().sum()

# Set city column as title
customersdf["customer_city"] = customersdf["customer_city"].str.title()

In [ ]:
# Load sellers table
sellersdf = pd.read_csv("D:/Downloads/Team Project/0_Data/olist_sellers_dataset.csv")

# Check table
sellersdf.info()

# Check null values
sellersdf.isnull().sum()

# Set city column as title
sellersdf["seller_city"] = sellersdf["seller_city"].str.title()

## Part 2 - Feature Engineering

In [ ]:
# Create processing time column
orderdf["processing_time"] = (orderdf["order_approved_at"] - orderdf["order_purchase_timestamp"]).dt.days

# Create seller_to_carrier_days column
orderdf['seller_to_carrier_days'] = (
    orderdf['order_delivered_carrier_date'] - orderdf['order_approved_at']).dt.days

# Create shipping_time column
orderdf['shipping_time'] = (
    orderdf['order_delivered_customer_date'] - orderdf['order_delivered_carrier_date']
).dt.days

# Create delivery_days column
orderdf['delivery_days'] = (
    orderdf['order_delivered_customer_date'] - orderdf['order_purchase_timestamp']
).dt.days

# Create delivery_delay_days column
orderdf['delivery_delay_days'] = (
    orderdf['order_delivered_customer_date'] - orderdf['order_estimated_delivery_date']
).dt.days

# Create delivery_status column
orderdf['delivery_status'] = np.where(
    orderdf['order_delivered_customer_date'] <= orderdf['order_estimated_delivery_date'],
    'On Time / Early',
    'Late Delivery'
)

In [ ]:
# Reorder the columns
orderdf = orderdf[[
    # 🟢 Order Info
    'order_id',
    'customer_id',
    'order_status',

    # 🟡 Timeline (chronological)
    'order_purchase_timestamp',
    'order_approved_at',
    'processing_time',

    'order_delivered_carrier_date',
    'seller_to_carrier_days',
    'shipping_time',
    
    'order_delivered_customer_date',
    'delivery_days',

    'order_estimated_delivery_date',

    # 🔴 Evaluation
    'delivery_delay_days',
    'delivery_status'
]]


## Part 3 - Analysis

## Part 4 - Visualization